<a href="https://colab.research.google.com/github/Decoding-Data-Science/CommunityWorkshops/blob/main/bootcampaug26/Day_2_rag_llamaindex_aug26_pdf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install llama_index

In [ ]:
pip install llama-index-readers-file

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 10.6 MB/s eta 0:00:00


In [ ]:
import openai
from google.colab import userdata

# Retrieve the OpenAI API key from Google Colab secrets
openai.api_key = userdata.get('openai')

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader

documents = SimpleDirectoryReader(
    input_dir="data",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

print(len(documents))
print(documents[0].text[:1000])

13
DDS Employee Handbook (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
Note: This document is a synthetic, training-friendly employee handbook for demos, onboarding
simulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified
counsel before any real-world use.
1. Welcome to Decoding Data Science (DDS)
DDS is a Dubai-based academy, consulting practice, and community focused on data science, AI, and
applied generative AI. We operate with a global mindset and a high trust culture—shipping practical
outcomes while supporting each other.
This handbook explains workplace expectations, benefits, and policies. If any local law conflicts with
this handbook, applicable law prevails.
2. Company Values & Ways of Working
 Build with clarity: define the user, problem, inputs/outputs, and definition of done.
 Bias for action: ship small, iterate fast, measure outcomes.
 Respect and inclusion: disagreement is allowed; disrespect is not.
 Dat

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
index = VectorStoreIndex.from_documents(documents=documents)
query_engine = index.as_query_engine()
response = query_engine.query("how to take sick leave?")
print(response)

Notify your line manager within 1 hour of your normal start time on the first day of absence by message or call. Log the absence in the leave system on the day of return, or earlier if able. A medical certificate from a licensed practitioner is required for any absence of 2 or more consecutive working days, and may be requested for shorter or repeated absences where a pattern suggests misuse. If the illness continues beyond the 15 fully paid days, additional sick leave follows the UAE Labour Law structure within the same leave year.


In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Configure LLM, Embedding, and Chunk Size
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# Define a system prompt
system_prompt = '''
You are Ayesha, an HR enterprise chatbot. Use only the four provided policy documents as your knowledge base to answer employee questions about HR topics. If information required to answer a question is not found within these documents, politely inform the employee to reach out to connect@decodingdatascience.com for further support, and do not make assumptions or provide outside information. If the question is not related to HR, politely clarify your HR-only scope and refrain from answering about non-HR topics. Always respond politely and cite the original document (by name and section, if possible) for any information or answer you provide, to enhance credibility.

Follow these steps for each question:

1. Understand the employee’s question and determine if it falls within HR topics.
2. Search the four policy documents for relevant information.
3. If an answer is found:
   - Respond politely, clearly, and concisely.
   - Include a source citation referencing the policy document name and section or page, if available.
4. If no answer can be found or the question is unrelated to HR:
   - Politely inform the employee. For missing answers, direct them to connect@decodingdatascience.com; for non-HR topics, state that you only handle HR-related queries.
5. Never guess, provide hypothetical information, or use knowledge outside the four policy documents.

**Output Format:**
Respond in a polite, professional tone. Provide the answer as a paragraph, always ending with the source citation (e.g., "Source: Employee Leave Policy, Section 4.2"). If the answer cannot be found or is outside of HR, provide the appropriate fallback response. Do not use code blocks or special formatting.

**Examples:**

**Example 1:**
*Employee Question:* What is the procedure for applying for annual leave?
*Response:*
To apply for annual leave, employees should submit an online leave application through the HR portal at least 2 weeks in advance. Approval from the direct manager is required before leave is granted. Source: Employee Leave Policy, Section 3.1.

**Example 2:**
*Employee Question:* Do you provide travel allowances?
*Response:*
Yes, eligible employees are provided with travel allowances as per the company guidelines. The allowance details and reimbursement procedures can be found in the Travel and Expense Policy, Section 2.1.

**Example 3:**
*Employee Question:* What is the process for requesting a new laptop?
*Response:*
This question is not covered by the HR policy documents I can reference. Please reach out to connect@decodingdatascience.com for assistance.

**Example 4:**
*Employee Question:* What is the capital of France?
*Response:*
I am an HR enterprise chatbot and can only assist with HR-related questions. Please let me know if you have any HR policy queries.

(For real interactions, responses may be longer and the citation should specifically reference the relevant section and document name. Replace all placeholders with the corresponding policy titles and sections.)

---

**Reminder:**
Your objective is to answer only from the four policy documents, always cite the source, never respond outside HR topics, and refer employees to connect@decodingdatascience.com if an answer is unavailable. Respond politely at all times.


'''



index = VectorStoreIndex.from_documents(documents=documents)

# Configure query engine with system prompt
query_engine = index.as_query_engine(system_prompt=system_prompt)

response = query_engine.query("What are decoding data science standard office hours in Dubai?")
print(response)

The standard office hours for Decoding Data Science in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.


In [ ]:
import gradio as gr
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# Configure LLM, Embedding, and Chunk Size
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")
Settings.chunk_size = 600
Settings.chunk_overlap = 200

# Load data and build the index
documents = SimpleDirectoryReader("data").load_data()
index = VectorStoreIndex.from_documents(documents=documents)
query_engine = index.as_query_engine()

# Function to handle queries
def query_document(query):
    response = query_engine.query(query)
    return str(response)

# Gradio interface
interface = gr.Interface(
    fn=query_document,
    inputs=gr.Textbox(label="Enter your query", placeholder="Type your question here..."),
    outputs=gr.Textbox(label="Response"),
    title="DDS Enterise Chatbot connect to Data",
    description="Ask questions about the documents loaded into the system."
)

# Launch the Gradio app
if __name__ == "__main__":
    interface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://839a9a91d6790af2f3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#storing the vector store in local file
import os
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

# check if storage already exists
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

# Either way we can now query the index
query_engine = index.as_query_engine()

retriever = VectorIndexRetriever(index=index, similarity_top_k=3)

query_engine = RetrieverQueryEngine(retriever=retriever)

response = query_engine.query("What are DDS standard office hours in Dubai?")
print(response)


The standard office hours in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.


In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader

documents = SimpleDirectoryReader(
    input_dir="data",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

print(len(documents))
print(documents[0].text[:1000])

9
DDS Employee Handbook (Synthetic) v1
Effective date: March 03, 2026  Dubai (GST)
Note: This document is a synthetic, training-friendly employee handbook for demos, onboarding
simulations, and HR-policy chatbot prototypes. It is not legal advice and must be reviewed by qualified
counsel before any real-world use.
1. Welcome to Decoding Data Science (DDS)
DDS is a Dubai-based academy, consulting practice, and community focused on data science, AI, and
applied generative AI. We operate with a global mindset and a high trust culture—shipping practical
outcomes while supporting each other.
This handbook explains workplace expectations, benefits, and policies. If any local law conflicts with
this handbook, applicable law prevails.
2. Company Values & Ways of Working
 Build with clarity: define the user, problem, inputs/outputs, and definition of done.
 Bias for action: ship small, iterate fast, measure outcomes.
 Respect and inclusion: disagreement is allowed; disrespect is not.
 Data

In [ ]:
#timing
import os
import time
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

# Start timer for index setup
start_time = time.time()

# check if storage already exists
PERSIST_DIR = "./storage"
if not os.path.exists(PERSIST_DIR):
    # load the documents and create the index
    documents = SimpleDirectoryReader("data").load_data()
    index = VectorStoreIndex.from_documents(documents)
    # store it for later
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # load the existing index
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

setup_duration = time.time() - start_time
print(f"Index setup time: {setup_duration:.2f} seconds")

# Start timer for query
query_start_time = time.time()

# Prepare the query engine
retriever = VectorIndexRetriever(index=index, similarity_top_k=2)
query_engine = RetrieverQueryEngine(retriever=retriever)

# Execute query
response = query_engine.query("Who all mentioned in the doc?")
print(response)

query_duration = time.time() - query_start_time
print(f"Query time: {query_duration:.2f} seconds")


Index setup time: 0.35 seconds
The document does not mention specific individuals by name. It primarily outlines policies, procedures, and guidelines relevant to employees at DDS.
Query time: 1.23 seconds


In [ ]:
import os
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)

import time
start_time = time.time()
query_engine = index.as_query_engine()

retriever = VectorIndexRetriever(index=index, similarity_top_k=2)

query_engine = RetrieverQueryEngine(retriever=retriever)

response = query_engine.query("What are DDS standard office hours in Dubai??")
print(response)


end_time = time.time()  # Record end time
execution_time = end_time - start_time  # Calculate execution time
print(f"Execution time: {execution_time} seconds")

The standard office hours for DDS in Dubai are from 9:00 AM to 6:00 PM, Monday to Friday.
Execution time: 0.8047778606414795 seconds
